In [11]:
pip install nltk Sastrawi

Note: you may need to restart the kernel to use updated packages.


## Import Library

In [12]:
import pandas as pd
import re
import nltk

from nltk.tokenize import word_tokenize
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## Load Dataset

In [5]:
df = pd.read_excel("datasetFeatureExtraction.xlsx")

df.head()

,created_at,full_text
0,Sun Mar 29 23:59:45 +0000 2026,@witasyahila @Hnirankara Anda layak jadi Owner...
1,Sun Mar 29 23:57:59 +0000 2026,Aldis Burger &gt;&gt;&gt; MBG
2,Sun Mar 29 23:57:47 +0000 2026,Kira2 Apa Komentar Menteri Hukum Prof Yusril t...
3,Sun Mar 29 23:57:14 +0000 2026,Iya sangat terasa sekali gak ada makanan bergi...
4,Sun Mar 29 23:55:51 +0000 2026,@Heraloebss MBG &gt;&gt;&gt; Mendem Bareng Gaess


### Inisialisasi Sastrawi

In [13]:
stop_factory = StopWordRemoverFactory()
stopword_remover = stop_factory.create_stop_word_remover()

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

## Membuat Fungsi Cleaning

In [20]:
slang_dict = {
    'yg': 'yang',
    'bgt': 'banget',
    'mbg': 'makan bergizi gratis', 
    'jd': 'jadi',
    'aja': 'saja',
    'sdh': 'sudah',
    'kan': '',
    'bukan': 'bukan'
}

def clean_text(text):
    # --- PRE-PROCESSING DENGAN REGEX ---
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # --- NORMALISASI SLANG ---
    # Memecah teks, ganti jika ada di dict, gabung lagi
    tokens = text.split()
    normalized_tokens = [slang_dict.get(word, word) for word in tokens]
    text = " ".join(normalized_tokens)
    
    # --- PENGOLAHAN BAHASA INDONESIA ---
    # Stopword removal (menggunakan Sastrawi yang sudah di-init di awal)
    text = stopword_remover.remove(text)
    
    # Stemming
    text = stemmer.stem(text)
    
    # Final cleaning (menghapus spasi ganda dan kata kosong)
    text = ' '.join(text.split())
    
    return text

# Menjalankan fungsi
print("Proses sedang berjalan...")
df['cleaned_text'] = df['full_text'].apply(clean_text)

# Set tampilan pandas
pd.set_option('display.max_colwidth', None)

# Lihat Hasil
print(df[['full_text', 'cleaned_text']].head())

Proses sedang berjalan...
                                                                                                       full_text  \
0                                                          @witasyahila @Hnirankara Anda layak jadi Owner MBG...   
1                                                                                  Aldis Burger &gt;&gt;&gt; MBG   
2  Kira2 Apa Komentar Menteri Hukum Prof Yusril tentang Carut Marut Proyek MBG dg Anggaran Ratusan Trilyun ini ?   
3                      Iya sangat terasa sekali gak ada makanan bergizi Selama MBG Libur https://t.co/rNrihyVcmp   
4                                                               @Heraloebss MBG &gt;&gt;&gt; Mendem Bareng Gaess   

                                                                                               cleaned_text  
0                                                                        layak jadi owner makan gizi gratis  
1                                                        

In [21]:
df.head()

,created_at,full_text,cleaned_text
0,Sun Mar 29 23:59:45 +0000 2026,@witasyahila @Hnirankara Anda layak jadi Owner MBG...,layak jadi owner makan gizi gratis
1,Sun Mar 29 23:57:59 +0000 2026,Aldis Burger &gt;&gt;&gt; MBG,aldis burger gt gt gt makan gizi gratis
2,Sun Mar 29 23:57:47 +0000 2026,Kira2 Apa Komentar Menteri Hukum Prof Yusril tentang Carut Marut Proyek MBG dg Anggaran Ratusan Trilyun ini ?,kira apa komentar menteri hukum prof yusril carut marut proyek makan gizi gratis dg anggar ratus trilyun
3,Sun Mar 29 23:57:14 +0000 2026,Iya sangat terasa sekali gak ada makanan bergizi Selama MBG Libur https://t.co/rNrihyVcmp,iya sangat asa sekali gak makan gizi lama makan gizi gratis libur
4,Sun Mar 29 23:55:51 +0000 2026,@Heraloebss MBG &gt;&gt;&gt; Mendem Bareng Gaess,makan gizi gratis gt gt gt mendem bareng gaess


### Representasi BOW

In [22]:
print("--- Bag of Words (Top 10) ---")
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(df['cleaned_text'])

# Menghitung frekuensi kata
bow_counts = pd.DataFrame({
    'word': bow_vectorizer.get_feature_names_out(),
    'count': bow_matrix.toarray().sum(axis=0)
})
print(bow_counts.sort_values(by='count', ascending=False).head(10))
print("\n")

--- Bag of Words (Top 10) ---
       word  count
600   makan    190
342    gizi    185
347  gratis    181
425    jadi     25
367    hari     17
177    buat     16
827  rakyat     15
44     anak     13
179   bukan     13
231   dapur     13




### TF-IDF

In [23]:
print("--- TF-IDF (Top 10) ---")
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['cleaned_text'])

# Menghitung rata-rata nilai TF-IDF untuk setiap kata
tfidf_scores = pd.DataFrame({
    'word': tfidf_vectorizer.get_feature_names_out(),
    'score': tfidf_matrix.toarray().mean(axis=0)
})
print(tfidf_scores.sort_values(by='score', ascending=False).head(10))
print("\n")

--- TF-IDF (Top 10) ---
          word     score
600      makan  0.089778
342       gizi  0.087446
347     gratis  0.086961
280  efisiensi  0.025219
904     sesuai  0.023619
367       hari  0.022092
425       jadi  0.021120
97      banget  0.017570
177       buat  0.014631
680         my  0.014511




### N-Gram

In [24]:
print("--- Bigram / N-gram (Top 10) ---")
# ngram_range=(2, 2) berarti kita mengambil 2 kata yang berdampingan (Bigram)
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
bigram_matrix = bigram_vectorizer.fit_transform(df['cleaned_text'])

bigram_counts = pd.DataFrame({
    'bigram': bigram_vectorizer.get_feature_names_out(),
    'count': bigram_matrix.toarray().sum(axis=0)
})
print(bigram_counts.sort_values(by='count', ascending=False).head(10))

--- Bigram / N-gram (Top 10) ---
               bigram  count
1135       makan gizi    179
561       gizi gratis    178
390       dapur makan     12
462   efisiensi makan      9
729        hari makan      9
663     gratis sesuai      8
343         ceo makan      8
1640     sesuai makan      8
1461    program makan      7
686             gt gt      4
